In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit, broadcast
spark = SparkSession.builder.appName("homework").getOrCreate()

24/12/08 18:58:34 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
#  - Disabled automatic broadcast join with `spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")`
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [3]:
#   - Explicitly broadcast JOINs `medals` and `maps`

df_medals = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/medals.csv") \
            .withColumnRenamed('name', 'medals_name') \
            .withColumnRenamed('description', 'medals_description')    
        
df_maps = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/maps.csv") \
            .withColumnRenamed('name', 'map_name') \
            .withColumnRenamed('description', 'map_description') \
            .withColumnRenamed('mapid', 'mapid')

df_medals_maps_broadcast = df_medals.join(broadcast(df_maps))

df_medals_maps_broadcast.show()

+----------+----------+-----------+----------+------------------+-------------------+------------+-------------+--------------+------------------+-----------+----------+--------------------+-------------------+--------------------+
|  medal_id|sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|classification|medals_description|medals_name|difficulty|               mapid|           map_name|     map_description|
+----------+----------+-----------+----------+------------------+-------------------+------------+-------------+--------------+------------------+-----------+----------+--------------------+-------------------+--------------------+
|2315448068|      NULL|       NULL|      NULL|              NULL|               NULL|        NULL|         NULL|          NULL|              NULL|       NULL|      NULL|c93d708f-f206-11e...|              Urban|Andesia was the c...|
|2315448068|      NULL|       NULL|      NULL|              NULL|       

In [4]:
#   - Bucket join `match_details`, `matches`, and `medal_matches_players` on `match_id` with `16` buckets

df_match_details = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/match_details.csv")

df_matches = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/matches.csv")

df_medal_matches_players = spark.read.option("header", "true") \
            .csv("/home/iceberg/data/medals_matches_players.csv")

In [5]:
spark.sql(""" CREATE DATABASE IF NOT EXISTS bootcamp""")

num_buckets = 16

In [6]:
spark.catalog.listDatabases()

[Database(name='bootcamp', catalog='demo', description=None, locationUri='s3://warehouse/bootcamp')]

In [7]:
spark.catalog.listTables('bootcamp')

[Table(name='actor_films', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='actors', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='match_details_bucketed', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='matches_bucketed', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='medal_matches_players_bucketed', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='resultUnsorted', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False),
 Table(name='sorted_by_playlist_id', catalog='demo', namespace=['bootcamp'], description=None, tableType='MANAGED', isTemporary=False)]

In [8]:
spark.sql(""" drop table if exists bootcamp.match_details_bucketed """)

bucketed_match_details_DDL = """
CREATE TABLE IF NOT EXISTS bootcamp.match_details_bucketed (
    match_id STRING
    ,player_gamertag STRING
    ,previous_spartan_rank int
    ,spartan_rank int
    ,previous_total_xp int
    ,total_xp int
    ,previous_csr_tier int
    ,previous_csr_designation int
    ,previous_csr int
    ,previous_csr_percent_to_next_tier int
    ,previous_csr_rank int
    ,current_csr_tier int
    ,current_csr_designation int
    ,current_csr int
    ,current_csr_percent_to_next_tier int
    ,current_csr_rank int
    ,player_rank_on_team int
    ,player_finished boolean
    ,player_average_life int
    ,player_total_kills int
    ,player_total_headshots int
    ,player_total_weapon_damage int
    ,player_total_shots_landed int
    ,player_total_melee_kills int
    ,player_total_melee_damage int
    ,player_total_assassinations int
    ,player_total_ground_pound_kills int
    ,player_total_shoulder_bash_kills int
    ,player_total_grenade_damage int
    ,player_total_power_weapon_damage int
    ,player_total_power_weapon_grabs int
    ,player_total_deaths int
    ,player_total_assists int
    ,player_total_grenade_kills int
    ,did_win int
    ,team_id int
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketed_match_details_DDL)

df_match_details.write.bucketBy(num_buckets, "match_id").saveAsTable("bootcamp.match_details_bucketed", format="parquet", mode="overwrite")

24/12/08 18:58:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
spark.sql(""" drop table if exists bootcamp.matches_bucketed """)

bucketed_matches_DDL = """
CREATE TABLE IF NOT EXISTS bootcamp.matches_bucketed (
    match_id STRING
    ,mapid STRING
    ,is_team_game boolean
    ,playlist_id STRING
    ,game_variant_id STRING
    ,is_match_over boolean
    ,completion_date timestamp
    ,match_duration STRING
    ,game_mode STRING
    ,map_variant_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketed_matches_DDL)

df_matches.write.bucketBy(16, "match_id").saveAsTable("bootcamp.matches_bucketed", format="parquet", mode="overwrite")

In [10]:
spark.sql(""" drop table if exists bootcamp.medal_matches_players_bucketed """)

bucketed_medal_matches_players_DDL = """
CREATE TABLE IF NOT EXISTS bootcamp.medal_matches_players_bucketed (
   match_id STRING,
   player_gamertag STRING,
   medal_id INTEGER,
   count INTEGER
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""
spark.sql(bucketed_medal_matches_players_DDL)

df_medal_matches_players.write.bucketBy(num_buckets, "match_id").saveAsTable("bootcamp.medal_matches_players_bucketed", format="parquet", mode="overwrite")

In [11]:
bucketed_match_details = spark.table("bootcamp.match_details_bucketed")
bucketed_matches = spark.table("bootcamp.matches_bucketed")
bucketed_medal_matches_players = spark.table("bootcamp.medal_matches_players_bucketed")


In [12]:
bucketed_matches.show()

+--------------------+--------------------+------------+--------------------+--------------------+-------------+--------------------+--------------+---------+--------------------+
|            match_id|               mapid|is_team_game|         playlist_id|     game_variant_id|is_match_over|     completion_date|match_duration|game_mode|      map_variant_id|
+--------------------+--------------------+------------+--------------------+--------------------+-------------+--------------------+--------------+---------+--------------------+
|f44c9997-eb6f-4d6...|ce1dc2de-f206-11e...|        true|0504ca3c-de41-48f...|b0df8938-0fb6-42e...|         true|2016-02-28 00:00:...|          NULL|     NULL|d5a6277a-96d5-499...|
|f0f2daf2-52f3-4ff...|cbcea2c0-f206-11e...|        NULL|2323b76a-db98-4e0...|257a305e-4dd3-41f...|         NULL|2016-02-04 00:00:...|          NULL|     NULL|7108c409-6d1e-41d...|
|8aec419e-2bfa-4fc...|c7edbf0f-f206-11e...|        true|f72e0ef0-7c4a-430...|1e473914-46e4-408...|  

In [13]:

# Read df2 and join with bucketed df1
result = (bucketed_match_details.join(bucketed_matches, "match_id")
            .join(bucketed_medal_matches_players, "match_id")
            .join(df_medals, "medal_id")
            .join(df_maps, "mapid")          
            .select(bucketed_match_details["*"], bucketed_matches["mapid"], bucketed_matches["is_team_game"], bucketed_matches["playlist_id"]
                , bucketed_matches["game_variant_id"], bucketed_matches["is_match_over"], bucketed_matches["completion_date"]
                , bucketed_matches["match_duration"], bucketed_matches["game_mode"], bucketed_matches["map_variant_id"]
                , bucketed_medal_matches_players["medal_id"], bucketed_medal_matches_players["count"]
                , df_maps["map_name"], df_maps["map_description"]
                , df_medals["classification"], df_medals["difficulty"]
                , df_medals["medals_description"], df_medals["medals_name"], df_medals["sprite_height"]
                , df_medals["sprite_left"], df_medals["sprite_sheet_height"], df_medals["sprite_sheet_width"]
                , df_medals["sprite_top"], df_medals["sprite_uri"], df_medals["sprite_width"]
            )
         )

        # .join(matchDetailsBucketTable, Seq("match_id"))
        # .join(medalMatchesPlayersExpandedBucketTable, Seq("match_id"), "left")

In [14]:
result.show()

+--------------------+---------------+---------------------+------------+-----------------+--------+-----------------+------------------------+------------+---------------------------------+-----------------+----------------+-----------------------+-----------+--------------------------------+----------------+-------------------+---------------+-------------------+------------------+----------------------+--------------------------+-------------------------+------------------------+-------------------------+---------------------------+-------------------------------+--------------------------------+---------------------------+--------------------------------+-------------------------------+-------------------+--------------------+--------------------------+-------+-------+--------------------+------------+--------------------+--------------------+-------------+--------------------+--------------+---------+--------------------+----------+-----+--------+--------------------+-------------

In [15]:
from pyspark.sql.functions import avg
#   - Aggregate the joined data frame to figure out questions like:
#     - Which player averages the most kills per game?

#group_data = result.groupBy(["player_gamertag", "match_id"])
#group_data.agg({'player_total_kills':'avg'}).show()

avg_kills = result.groupBy(["player_gamertag", "match_id"]).agg(avg("player_total_kills").alias("avg_kills"))

# most_kills = avg_kills.agg({"avg_kills": "max"}).collect()[0][0]
most_kills = avg_kills.orderBy("avg_kills", ascending=False).limit(1)

most_kills.show()

#print("player averages the most kills per game:", most_kills)

+---------------+--------------------+---------+
|player_gamertag|            match_id|avg_kills|
+---------------+--------------------+---------+
|   gimpinator14|acf0e47e-20ac-4b1...|    109.0|
+---------------+--------------------+---------+



In [16]:
from pyspark.sql.functions import count
#     - Which playlist gets played the most?

agg_playlist = result.groupBy(["playlist_id"]).agg(count("playlist_id").alias("agg_playlist_id"))

most_played = agg_playlist.orderBy("agg_playlist_id", ascending=False).limit(1)

most_played.show()

+--------------------+---------------+
|         playlist_id|agg_playlist_id|
+--------------------+---------------+
|f72e0ef0-7c4a-430...|        1565529|
+--------------------+---------------+



In [17]:
from pyspark.sql.functions import count
#     - Which map gets played the most?

agg_map_id = result.groupBy(["mapid", "map_name"]).agg(count("mapid").alias("agg_mapid"))

most_map = agg_map_id.orderBy("agg_mapid", ascending=False).limit(1)

most_map.show()

+--------------------+--------+---------+
|               mapid|map_name|agg_mapid|
+--------------------+--------+---------+
|c74c9d0f-f206-11e...|  Alpine|  1445545|
+--------------------+--------+---------+



In [18]:
from pyspark.sql.functions import sum
#     - Which map do players get the most Killing Spree medals on?
medal_map = (result.filter("classification == 'KillingSpree'")
             .groupBy(["mapid", "map_name"]).agg(count("mapid").alias("map_kills"))
             .select("map_name", "mapid","map_kills"))

most_kills_map = medal_map.orderBy("map_kills", ascending=False).limit(1)

most_kills_map.show()

+--------+--------------------+---------+
|map_name|               mapid|map_kills|
+--------+--------------------+---------+
|  Alpine|c74c9d0f-f206-11e...|    64309|
+--------+--------------------+---------+



In [19]:
medal_map.orderBy("map_kills", ascending=False).show()

+--------------+--------------------+---------+
|      map_name|               mapid|map_kills|
+--------------+--------------------+---------+
|        Alpine|c74c9d0f-f206-11e...|    64309|
|Breakout Arena|c7edbf0f-f206-11e...|    51960|
|       Glacier|c7805740-f206-11e...|    40845|
|        Empire|cdb934b0-f206-11e...|    15914|
|         Truth|ce1dc2de-f206-11e...|    13934|
|      Coliseum|cebd854f-f206-11e...|    13737|
|       The Rig|cb914b9e-f206-11e...|    13642|
|         Plaza|caacb800-f206-11e...|    13094|
|          Eden|cd844200-f206-11e...|    12342|
|        Regret|cdee4e70-f206-11e...|    12033|
|        Fathom|cc040aa1-f206-11e...|    11652|
|      Parallax|c7b7baf0-f206-11e...|     8165|
|    Overgrowth|ca737f8f-f206-11e...|     7113|
|          NULL|cc74f4e1-f206-11e...|     6458|
|      Riptide |cbcea2c0-f206-11e...|     6418|
|          NULL|ce89a40f-f206-11e...|     2404|
+--------------+--------------------+---------+



In [20]:
#   - With the aggregated data set
#     - Try different `.sortWithinPartitions` to see which has the smallest data size (hint: playlists and maps are both very low cardinality)
result.rdd.getNumPartitions()

20

In [21]:
result.write.mode("overwrite").saveAsTable("bootcamp.resultUnsorted")

In [ ]:
result.sortWithinPartitions("playlist_id").write.mode("append").saveAsTable("bootcamp.sorted_by_playlist_id")

In [24]:
result.sortWithinPartitions("mapid").write.mode("append").saveAsTable("bootcamp.sorted_by_mapid")

In [26]:
%%sql
SELECT
        SUM(file_size_in_bytes) as total_size
    ,   COUNT(1) AS number_of_files
    ,   'no sorting'
FROM bootcamp.resultUnsorted.files
UNION ALL
SELECT
        SUM(file_size_in_bytes) as total_size
    ,   COUNT(1) AS number_of_files
    ,   'sorted_by_playlist_id'
FROM bootcamp.sorted_by_playlist_id.files
UNION ALL
SELECT
        SUM(file_size_in_bytes) as total_size
    ,   COUNT(1) AS number_of_files
    ,   'sorted_by_mapid'
FROM bootcamp.sorted_by_mapid.files

total_size,number_of_files,no sorting
151626803,20,no sorting
147392783,20,sorted_by_playlist_id
150102931,20,sorted_by_mapid
